In [13]:
import torch as T

from rl_lib.envs.make_env import make_env
from rl_lib.agent import Agent
from rl_lib.networks.factory import Network

In [14]:
DEVICE = T.device("cuda" if T.cuda.is_available() else "cpu")
STACK_SIZE = 4
SKIP = 2

In [15]:
env = make_env(
    "CarRacing-v3",
    num_envs=1,
    skip=SKIP,
    record=False,
    wrappers=[
        "record_episode_stats",
        "grayscale",
        "max_and_skip",
    ]
)

In [16]:
network = Network(
    env.observation_space.shape[-1],
    env.action_space.shape[-1],
    STACK_SIZE
).to(DEVICE)

In [17]:
agent = Agent(
    network=network,
    observation_dim=env.observation_space.shape[-1],
    action_dim=env.action_space.shape[-1],
    stack_size=STACK_SIZE,
    device=DEVICE
)

In [18]:
agent.load_state_dict("/workspace/TrackmaniaRL/logs/checkpoints/checkpoint_400.pt")

In [41]:
agent.train()
state, _ = env.reset()
done = T.zeros(env.num_envs, dtype=T.bool).to(agent.device)
print(state.shape, type(state), done)

(1, 96, 96, 1) <class 'numpy.ndarray'> tensor([False], device='cuda:0')


In [42]:
for _ in range(4):
    state, _, _, _, _, _, _, _, done, info = agent.step_env(env, state, done, temperature=1e-4)

In [43]:
state.max(), state.min(), state.shape

(np.uint8(208), np.uint8(0), (1, 96, 96, 1))

In [44]:
state_t = T.from_numpy(state).to(agent.device)
features = agent.feature_extract(state_t)
print(features)

tensor([[ 0.8871, -0.3664, -1.2322,  0.5660,  0.4050, -1.0343,  0.0027,  1.0028,
         -1.0379,  0.3530,  0.4179, -0.0831,  0.6107, -0.5632,  0.0152, -0.7995,
         -0.5117, -0.2762,  0.9583, -0.8609,  0.2934,  0.5294,  0.8320,  0.6443,
         -0.2425, -0.2291,  0.6305,  0.2834, -0.1168, -1.1899,  0.1652,  1.0971,
         -0.6005,  0.5260,  0.1260, -0.2415,  0.4715, -0.1671,  1.4499,  0.4233,
         -0.3596, -0.5052,  0.4707, -0.8173,  0.3448,  0.8164,  0.1115, -0.4768,
         -0.0454, -0.6539, -0.3854,  0.9683,  0.1158, -0.4116,  0.1084,  0.7423,
          0.1873, -0.5131,  0.3404,  0.1539,  0.5247,  0.9528, -0.1285, -0.3860,
         -0.2753,  1.0513, -0.2851, -0.8540, -0.0357,  0.0744,  0.2597, -0.5259,
         -0.5039,  0.1669,  0.5775, -0.0291,  0.8636,  0.3131, -0.3797,  0.2225,
          0.4283,  0.5958, -0.0747, -0.4694,  0.3568, -0.7938,  0.2553, -0.7562,
          0.9069,  0.4456,  0.6986, -0.4031,  0.1833,  0.4555, -0.5636,  0.0946,
         -0.3494,  0.0031, -

In [45]:
state_t.shape

torch.Size([1, 96, 96, 1])

In [46]:
def run_layers(network, x):
    for name, layer in network.named_children():
        x = layer(x)

        print(
            f"{name}: "
            f"shape={tuple(x.shape)}, "
            f"mean={x.mean().item():.4f}, "
            f"std={x.std().item():.4f}, "
            f"min={x.min().item():.4f}, "
            f"max={x.max().item():.4f}, "
            f"zero={(x == 0).float().mean().item():.2%}"
        )

    return x


t = run_layers(agent._network.cnn._network, agent._preprocess_observation(state_t))

0: shape=(1, 32, 23, 23), mean=0.0126, std=0.9631, min=-2.9292, max=3.6257, zero=0.00%
1: shape=(1, 32, 23, 23), mean=0.3840, std=0.5738, min=0.0000, max=3.6257, zero=47.65%
2: shape=(1, 64, 10, 10), mean=0.0676, std=0.8629, min=-4.6378, max=3.9991, zero=0.00%
3: shape=(1, 64, 10, 10), mean=0.3519, std=0.5567, min=0.0000, max=3.9991, zero=49.48%
4: shape=(1, 64, 8, 8), mean=0.0069, std=0.5923, min=-2.7237, max=3.5232, zero=0.00%
5: shape=(1, 64, 8, 8), mean=0.2170, std=0.3686, min=0.0000, max=3.5232, zero=48.54%
6: shape=(1, 4096), mean=0.2170, std=0.3686, min=0.0000, max=3.5232, zero=48.54%
7: shape=(1, 256), mean=-0.3400, std=0.7230, min=-2.1731, max=2.0816, zero=0.00%
8: shape=(1, 256), mean=0.1607, std=0.3309, min=0.0000, max=2.0816, zero=71.48%
9: shape=(1, 256), mean=-0.0103, std=0.5940, min=-1.8119, max=1.6010, zero=0.00%


In [47]:
temporal = agent.temporal_encode(
    agent._get_features_window(features),
    agent._get_mask_window(done)
)

In [49]:
temporal.max(), temporal.min(), temporal.std()

(tensor(3.7737, device='cuda:0', grad_fn=<MaxBackward1>),
 tensor(-4.3264, device='cuda:0', grad_fn=<MinBackward1>),
 tensor(1.8626, device='cuda:0', grad_fn=<StdBackward0>))